<a href="https://colab.research.google.com/github/ArjunHirani/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Raw table grain:** one row in `fact_content_daily_performance` = one `(report_date, client_hash_id, content_hash_id)` observation — a single content item's daily performance for one client on one day.

**My lane's slice:** one row in my built feature frame = one `(client_hash_id, content_hash_id)` content item, with features aggregated over **March 2026** (the feature/decision window) and a label measured from **April 2026** (the following, future window). This is a genuine past→future split — March is fully in the past relative to the April outcome, continuing the discipline from my Week 1-2 framing notebooks.

I deliberately work on `month='2026-03'` as my mid-panel iteration month, per the card's warning that the `_sample` table is the sealed final month (June 2026) and must never be used to build label logic.

In [1]:
%pip -q install duckdb huggingface_hub

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [HF_TOKEN])

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


Paste your Hugging Face READ token (hf_...): ··········
dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature** (knowable by end of March, safe to use):
- `impressions_march` — total GSC impressions, March, counted only where `gsc_data_available IS TRUE`
- `clicks_march` — total GSC clicks, March, same availability filter
- `avg_position_march` — average GSC position, March
- `active_days_march` — count of distinct days with confirmed GSC data in March (a coverage/consistency signal)
- `content_age_days_march` — days between `content_created_date` (from `dim_content`) and March 31 — a static property, safe regardless of window

**Label / proxy:** `is_declining_april` = 1 if total April clicks < total March clicks, else 0. Built **entirely from April data** — never touches a March column, so it structurally cannot leak backward into the features.

**Context** (never a feature): `client_hash_id`, `content_hash_id` — grouping/joining only, and critically, used for a **client-grouped** split, not just a random one (see Section 3 caveat).

**Excluded, with why:**
- `ga4_*` columns (pageviews, sessions, engagement) — **excluded from this pass entirely**. Verified below: GA4 availability in March is only ~4.2% of rows, far too sparse to build a reliable feature at this grain yet.
- `gsc_sum_position` — redundant with `gsc_avg_position`, and summing a per-day average across days isn't a meaningful quantity on its own.
- Individual `ai_chatgpt` / `ai_perplexity` / etc. columns — too sparse at single-content, single-month grain (the card's own AI-referral caution); not used even in aggregate this pass.
- `fact_content_query_90d` (query-level detail) — excluded for this contract pass; its 90-day window doesn't cleanly align with my March/April split without extra care, so I'm deferring it rather than risking a window-overlap mistake.

In [2]:
field_buckets = {
    "feature": ["impressions_march", "clicks_march", "avg_position_march",
                 "active_days_march", "content_age_days_march"],
    "label": ["is_declining_april (derived from clicks_april < clicks_march)"],
    "context": ["client_hash_id", "content_hash_id"],
    "excluded": ["ga4_* columns (too sparse, ~4.2% availability in March)",
                  "gsc_sum_position (redundant/meaningless alone)",
                  "individual ai_* columns (too sparse per-content)",
                  "fact_content_query_90d (window alignment not yet verified)"]
}
for bucket, fields in field_buckets.items():
    print(f"{bucket.upper()}:")
    for f in fields:
        print(f"  - {f}")
    print()

FEATURE:
  - impressions_march
  - clicks_march
  - avg_position_march
  - active_days_march
  - content_age_days_march

LABEL:
  - is_declining_april (derived from clicks_april < clicks_march)

CONTEXT:
  - client_hash_id
  - content_hash_id

EXCLUDED:
  - ga4_* columns (too sparse, ~4.2% availability in March)
  - gsc_sum_position (redundant/meaningless alone)
  - individual ai_* columns (too sparse per-content)
  - fact_content_query_90d (window alignment not yet verified)



## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Three verification queries**, all on `month='2026-03'`:
1. **Grain** — group by the stated unit columns and check nothing has `COUNT(*) > 1`.
2. **Counts + date span** — total rows, min/max date, distinct clients and content items, compared against what the dataset card promised.
3. **Availability** — filtered explicitly with `IS TRUE` (not just truthy), showing what share of rows actually survive.

In [3]:
# Query 1: GRAIN -- does one row really equal one (date, client, content)?
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()
print("Rows violating the stated grain (should be EMPTY):")
print(grain_check)

# Query 2: ROW COUNT + DATE SPAN for the mid-panel month
counts = con.sql(f"""
    SELECT COUNT(*) AS row_count,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()
print("\nRow count + date span for month=2026-03:")
print(counts)

# Query 3: AVAILABILITY -- filter with IS TRUE, show survival rate
availability = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()
availability['gsc_available_share'] = availability['gsc_available_rows'] / availability['total_rows']
availability['ga4_available_share'] = availability['ga4_available_rows'] / availability['total_rows']
print("\nAvailability check (month=2026-03):")
print(availability)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows violating the stated grain (should be EMPTY):
Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, c]
Index: []

Row count + date span for month=2026-03:
   row_count   min_date   max_date  n_clients  n_content
0    9841378 2026-03-01 2026-03-31         55     331437


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Availability check (month=2026-03):
   total_rows  gsc_available_rows  ga4_available_rows  gsc_available_share  \
0     9841378           3611061.0            413966.0             0.366926   

   ga4_available_share  
0             0.042064  


**Reading these results:** the grain holds (zero violations). March 2026 has 9,841,378 rows spanning the full month (2026-03-01 to 2026-03-31), across 55 active clients and 331,437 content items. Only **36.7%** of rows have confirmed GSC availability and just **4.2%** have confirmed GA4 availability — this is exactly why the feature build below filters with `gsc_data_available IS TRUE` rather than trusting raw non-null values, and why GA4 fields are excluded entirely for this pass.

### Five features + the leakage trap

Now I build the actual feature frame: March aggregates (features) joined to an April outcome (label), filtered to items with real March activity (>=100 impressions) so "declining" isn't confused with "no data."

In [4]:
import pandas as pd

data = con.sql(f"""
    WITH march AS (
        SELECT client_hash_id, content_hash_id,
               SUM(CASE WHEN gsc_data_available THEN gsc_impressions ELSE 0 END) AS impressions_march,
               SUM(CASE WHEN gsc_data_available THEN gsc_clicks ELSE 0 END) AS clicks_march,
               AVG(CASE WHEN gsc_data_available THEN gsc_avg_position END) AS avg_position_march,
               COUNT(DISTINCT CASE WHEN gsc_data_available THEN report_date END) AS active_days_march
        FROM {TABLES['fact_daily']}
        WHERE month = '2026-03'
        GROUP BY 1, 2
        HAVING impressions_march >= 100
    ),
    april AS (
        SELECT client_hash_id, content_hash_id,
               SUM(CASE WHEN gsc_data_available THEN gsc_clicks ELSE 0 END) AS clicks_april
        FROM {TABLES['fact_daily']}
        WHERE month = '2026-04'
        GROUP BY 1, 2
    )
    SELECT m.*, a.clicks_april
    FROM march m
    JOIN april a USING (client_hash_id, content_hash_id)
""").df()

print(f"Rows with real March activity AND an April outcome: {len(data):,}")

# Fifth feature: content age at end of March -- a static content property, from dim_content
content_meta = con.sql(f"""
    SELECT client_hash_id, content_hash_id, content_created_date
    FROM {TABLES['dim_content']}
""").df()

data = data.merge(content_meta, on=['client_hash_id', 'content_hash_id'], how='left')
data['content_age_days_march'] = (pd.Timestamp('2026-03-31') - pd.to_datetime(data['content_created_date'])).dt.days

# The label: built ENTIRELY from April data, never touches a March column
data['is_declining_april'] = (data['clicks_april'] < data['clicks_march']).astype(int)

feature_cols = ['impressions_march', 'clicks_march', 'avg_position_march',
                 'active_days_march', 'content_age_days_march']
print(f"\nLabel distribution: {data['is_declining_april'].value_counts().to_dict()}")
print(f"Decline rate: {data['is_declining_april'].mean():.3f}")
print(f"\nFeature frame preview:")
print(data[['client_hash_id','content_hash_id'] + feature_cols + ['clicks_april','is_declining_april']].head(8).to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows with real March activity AND an April outcome: 101,441


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Label distribution: {0: 60644, 1: 40797}
Decline rate: 0.402

Feature frame preview:
         client_hash_id          content_hash_id  impressions_march  clicks_march  avg_position_march  active_days_march  content_age_days_march  clicks_april  is_declining_april
client_62f4a7e64f5e0096 content_143987cfdeaba4c0              345.0           1.0           38.879856                 31                      56           0.0                   1
client_62f4a7e64f5e0096 content_86ab16840c4e0e1a              854.0           1.0            8.792961                 31                      56           1.0                   0
client_62f4a7e64f5e0096 content_3f87f49c36774e23              205.0           0.0           33.410077                 29                      56           0.0                   0
client_62f4a7e64f5e0096 content_64706c8afebebb8c              666.0           2.0            5.928198                 31                      56           1.0                   1
client_62f4a7e64f5e

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

model_data = data.dropna(subset=feature_cols + ['is_declining_april'])
X_clean = model_data[feature_cols]
y = model_data['is_declining_april']

X_tr, X_te, y_tr, y_te = train_test_split(X_clean, y, test_size=0.25, random_state=42, stratify=y)
honest_model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
honest_auc = roc_auc_score(y_te, honest_model.predict_proba(X_te)[:, 1])
print(f"HONEST model (5 clean March-only features) -- ROC-AUC: {honest_auc:.3f}")

# THE TRAP: add clicks_april itself as a "feature" -- it IS the answer in disguise
model_data_leaky = model_data.copy()
leaky_feature_cols = feature_cols + ['clicks_april']
X_leaky = model_data_leaky[leaky_feature_cols]

X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_leaky, y, test_size=0.25, random_state=42, stratify=y)
leaky_model = LogisticRegression(max_iter=1000).fit(X_tr2, y_tr2)
leaky_auc = roc_auc_score(y_te2, leaky_model.predict_proba(X_te2)[:, 1])
print(f"LEAKY model (adding clicks_april as a 'feature') -- ROC-AUC: {leaky_auc:.3f}  <- watch this jump")

print(f"\nDeleting the leaked column. Keeping the HONEST number: {honest_auc:.3f}")
del model_data_leaky, X_leaky, leaky_model

HONEST model (5 clean March-only features) -- ROC-AUC: 0.682
LEAKY model (adding clicks_april as a 'feature') -- ROC-AUC: 1.000  <- watch this jump

Deleting the leaked column. Keeping the HONEST number: 0.682


**The trap, performed on real warehouse data:** adding `clicks_april` — the exact column the label is derived from — as a "feature" sends ROC-AUC from a genuine **0.682** to a meaningless **1.000**. That perfect score isn't a good model, it's the label smuggled back in as an input. I deleted the leaked column and kept the honest 0.682 as the real number going forward.

**One more honest caveat, caught while writing this up:** the train/test split above is a **random row-level split**, not grouped by `client_hash_id`. That means a client's *other* content items can appear in both train and test — a softer, client-level leakage risk, distinct from the label leak but real. I'm flagging this now rather than presenting 0.682 as fully clean; a proper client-grouped split (`GroupShuffleSplit` on `client_hash_id`, as notebook 03's "your turn" suggests) is required before this number can be trusted for real modeling decisions in later weeks.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- **GA4 is nearly unusable at this grain right now.** Only ~4.2% of March rows have `ga4_data_available IS TRUE`. This isn't random missingness — it likely follows client-level GA4 access/rollout patterns (`dim_clients.ga4_data_start`), not a per-row coin flip. Any engagement-based feature would be built on a small, non-representative slice of clients, not the general population.
- **GSC availability is also partial (36.7%), not universal**, even though `client_has_gsc` may be true for a client overall — day-level availability still varies. Blind `fillna(0)` on GSC metrics would misrepresent "no confirmed data" as "zero activity," which is why every aggregate above filters with `gsc_data_available IS TRUE` explicitly.
- **The panel is unbalanced across clients.** March shows only 55 distinct clients out of 104 in `dim_clients` — many clients simply have no March data, likely due to differing `gsc_data_start`/`ga4_data_start` per client. Any per-client comparison must account for this rather than assuming a shared calendar window.
- **This is a two-month slice (March→April) of a ~17-month panel.** A single past→future pair can't tell me whether the decline pattern is seasonal, one-off, or persistent — that requires repeating this across multiple month-pairs before trusting it broadly.
- **The current train/test split is row-level random, not client-grouped** (see Section 3) — so the 0.682 ROC-AUC is a first honest read, not yet a fully leakage-safe one at the client level.
- **Query-level detail (`fact_content_query_90d`) is excluded from this pass** — its fixed 90-day window doesn't obviously align with a March/April monthly split, and forcing it in without checking that alignment first would risk a new leakage mistake.

In [6]:
# Supporting check for Section 4: confirm the panel really is unbalanced across clients
# (per dim_clients history-start columns), not just an artifact of my March filter.
history_check = con.sql(f"""
    SELECT COUNT(*) AS n_clients_total,
           SUM(CASE WHEN gsc_data_start IS NULL THEN 1 ELSE 0 END) AS n_no_gsc_start,
           MIN(gsc_data_start) AS earliest_gsc_start,
           MAX(gsc_data_start) AS latest_gsc_start,
           SUM(CASE WHEN ga4_data_start IS NULL THEN 1 ELSE 0 END) AS n_no_ga4_start
    FROM {TABLES['dim_clients']}
""").df()
print("Client history-start spread (confirms an unbalanced panel):")
print(history_check)

Client history-start spread (confirms an unbalanced panel):
   n_clients_total  n_no_gsc_start earliest_gsc_start latest_gsc_start  \
0              104            37.0         2025-01-27       2026-06-02   

   n_no_ga4_start  
0            53.0  
